<a href="https://colab.research.google.com/github/nurashilaharman-star/Tugas-1-Estimasi-Titik-dan-Interval/blob/main/251061014_Nur_Ashilah_Arman_Tugas_1_(AED).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Tugas 1 : Estimasi Titik dan Interval**

Pada tugas ini dilakukan analisis estimasi titik dan estimasi interval serta proses resampling menggunakan metode Jackknife dan Bootstrapping. Data yang digunakan adalah dataset Student Performance – Multiple Linear Regression yang diperoleh dari Kaggle. Dataset ini berisi informasi mengenai kebiasaan belajar dan performa siswa.

In [1]:
import pandas as pd

df = pd.read_csv('/content/Student_Performance.csv')
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


In [2]:
data = df[['Hours Studied',
           'Previous Scores',
           'Sleep Hours']].head(100).copy()

data.insert(0, 'No Sampel', range(1, 101))

data

,No Sampel,Hours Studied,Previous Scores,Sleep Hours
0,1,7,99,9
1,2,4,82,4
2,3,8,51,7
3,4,5,52,5
4,5,7,75,8
...,...,...,...,...
95,96,3,48,7
96,97,8,64,5
97,98,6,95,6
98,99,3,55,5


Dataset dibaca menggunakan library Pandas, kemudian dipilih beberapa kolom numerik yang akan digunakan dalam analisis.

Kolom yang digunakan meliputi:
1. Hours Studied: jumlah jam belajar siswa.
2. Previous Scores: nilai ujian sebelumnya.
3. Sleep Hours: jumlah jam tidur siswa.

Selanjutnya, diambil 100 sampel pertama dari dataset. Variabel Hours Studied ditetapkan sebagai variabel utama dalam seluruh proses perhitungan estimasi dan resampling.

In [3]:
print("Jumlah sampel:", len(data))
print("Jumlah kolom:", len(data.columns))
print("Data kosong:")
print(data.isnull().sum())

Jumlah sampel: 100
Jumlah kolom: 4
Data kosong:
No Sampel          0
Hours Studied      0
Previous Scores    0
Sleep Hours        0
dtype: int64


### **A. Menghitung nilai estimasi titik dan interval menggunakan    distribusi Z dengan tingkat kesalahan 5%.**

**1. Menghitung Estimasi Titik**

Estimasi titik merupakan proses memperkirakan parameter populasi menggunakan satu nilai yang diperoleh dari sampel.

Estimasi titik untuk rata-rata populasi dihitung menggunakan rata-rata sampel dengan rumus:

x̄ = Σxi / n

Keterangan:

• x̄ = rata-rata sampel.

• xi = nilai setiap pengamatan.

• n = jumlah sampel.

In [4]:
# Mengambil variabel utama
x = data['Hours Studied']

# Menghitung jumlah sampel
n = len(x)

# Menghitung rata-rata sampel
x_bar = x.mean()

print("Jumlah sampel:", n)
print("Estimasi titik:", x_bar)

Jumlah sampel: 100
Estimasi titik: 5.32


Berdasarkan hasil perhitungan menggunakan Python, diperoleh jumlah sampel sebanyak 100 dengan nilai estimasi titik sebesar 5,32.

Hasil tersebut menunjukkan bahwa rata-rata jam belajar dari 100 sampel yang digunakan adalah 5,32 jam per hari.

**2. Menghitung Estimasi Interval Z**

Pada tugas ini digunakan distribusi Z dengan tingkat kesalahan sebesar 5% atau α = 0,05. Dengan demikian, tingkat kepercayaan yang digunakan adalah 95% dan nilai Z tabel sebesar 1,96.

Rumus estimasi interval untuk rata-rata adalah:

x̄ ± Zα/2 (σ / √n)

Keterangan:

• x̄ = rata-rata sampel

• Zα/2 = nilai Z tabel.

• σ = simpangan baku populasi.

• n = jumlah sampel.

Karena simpangan baku populasi tidak diketahui, pada perhitungan ini digunakan simpangan baku sampel sebagai pendekatan

In [5]:
import numpy as np

# Menghitung simpangan baku sampel
s = x.std(ddof=1)

# Nilai Z untuk tingkat kepercayaan 95%
z = 1.96

# Menghitung standard error
SE = s / np.sqrt(n)

# Menghitung margin of error
ME = z * SE

# Menghitung batas interval
batas_bawah = x_bar - ME
batas_atas = x_bar + ME

print("Rata-rata sampel:", x_bar)
print("Simpangan baku:", s)
print("Standard Error:", SE)
print("Margin of Error:", ME)
print("Batas bawah:", batas_bawah)
print("Batas atas:", batas_atas)
print("Estimasi interval: (", batas_bawah, ",", batas_atas, ")")

Rata-rata sampel: 5.32
Simpangan baku: 2.639559191848655
Standard Error: 0.2639559191848655
Margin of Error: 0.5173536016023363
Batas bawah: 4.802646398397664
Batas atas: 5.837353601602336
Estimasi interval: ( 4.802646398397664 , 5.837353601602336 )


Berdasarkan hasil perhitungan diperoleh:

• Rata-rata sampel = 5,32.

• Simpangan baku sampel = 2,6396.

• Standard Error = 0,2640.

• Margin of Error = 0,5174.

• Batas bawah = 4,8026.

• Batas atas = 5,8374.

Dengan demikian, diperoleh estimasi interval sebesar 4,8026 sampai 5,8374 jam.

Hasil tersebut menunjukkan bahwa berdasarkan pendekatan interval yang digunakan, rata-rata jam belajar populasi diperkirakan berada dalam rentang tersebut dengan tingkat kepercayaan 95%.

### **B. proses resampling menggunakan metode jackknife dan bootstrapping, hitung nilai rata-rata dari hasil resamplingnya. Untuk bootstrapping lakukan iterasi sebanyak 1000.**

**1. Melakukan Proses Jacknife**

Jackknife merupakan salah satu metode resampling yang dilakukan dengan cara menghilangkan satu observasi secara bergantian dari keseluruhan data, kemudian menghitung kembali nilai estimasinya.
Proses ini dikenal dengan istilah leave-one-out. Karena jumlah data yang digunakan sebanyak 100 sampel, maka proses Jackknife dilakukan sebanyak 100 kali.

Setelah seluruh proses resampling selesai, dilakukan perhitungan rata-rata dari semua hasil resampling Jackknife.

In [6]:
import numpy as np

# Mengubah data menjadi array
data_jack = np.array(x)

# Menghitung jumlah sampel
n = len(data_jack)

# Menyimpan hasil resampling
jackknife_means = []

# Proses leave-one-out
for i in range(n):
    sampel = np.delete(data_jack, i)
    rata_rata = np.mean(sampel)
    jackknife_means.append(rata_rata)

# Menghitung rata-rata hasil Jackknife
rata_jackknife = np.mean(jackknife_means)

print("Jumlah resampling:", len(jackknife_means))
print("Rata-rata awal:", np.mean(data_jack))
print("Rata-rata Jackknife:", rata_jackknife)

Jumlah resampling: 100
Rata-rata awal: 5.32
Rata-rata Jackknife: 5.32


Berdasarkan hasil perhitungan menggunakan Python, diperoleh:

• Jumlah resampling = 100.

• Rata-rata awal = 5,32.

• Rata-rata Jackknife = 5,32.

Hasil tersebut menunjukkan bahwa rata-rata dari seluruh hasil resampling Jackknife sama dengan rata-rata sampel awal. Hal ini sesuai dengan karakteristik Jackknife ketika digunakan untuk mengestimasi rata-rata.

**2. Resampling dengan Metode Bootstrapping**

Bootstrapping merupakan metode resampling yang dilakukan dengan mengambil sampel secara acak dari data yang tersedia dengan pengembalian (with replacement).

Pada tugas ini, proses Bootstrapping dilakukan sebanyak 1.000 iterasi. Setiap iterasi mengambil kembali 100 data secara acak dari dataset awal.

Pada setiap iterasi dihitung nilai rata-rata, kemudian seluruh hasil rata-rata tersebut disimpan untuk mendapatkan rata-rata keseluruhan hasil Bootstrapping.

In [8]:
np.random.seed(42)

# Menentukan jumlah iterasi
B = 1000

# Menyimpan hasil resampling
bootstrap_means = []

# Proses Bootstrapping
for i in range(B):
    sampel_bootstrap = np.random.choice(
        x, size=len(x), replace=True
    )

    rata_rata = np.mean(sampel_bootstrap)
    bootstrap_means.append(rata_rata)

# Menghitung rata-rata hasil Bootstrapping
rata_bootstrap = np.mean(bootstrap_means)

print("Jumlah iterasi:", B)
print("Rata-rata awal:", np.mean(x))
print("Rata-rata Bootstrapping:", rata_bootstrap)

Jumlah iterasi: 1000
Rata-rata awal: 5.32
Rata-rata Bootstrapping: 5.32562


Berdasarkan hasil perhitungan menggunakan Python, diperoleh:

• Jumlah iterasi = 1.000

• Rata-rata awal = 5,32

• Rata-rata Bootstrapping = 5,32139

Hasil tersebut menunjukkan bahwa rata-rata Bootstrapping sangat dekat dengan rata-rata sampel awal. Hal ini memperlihatkan bahwa hasil estimasi rata-rata dari proses Bootstrapping mendekati nilai estimasi awal.

### **Kesimpulan**

• Berdasarkan seluruh proses analisis yang telah dilakukan terhadap 100 sampel data jam belajar siswa, diperoleh estimasi titik sebesar 5,32 jam dengan estimasi interval menggunakan distribusi Z sebesar 4,8026 sampai 5,8374 jam pada tingkat kepercayaan 95%.

• Proses resampling menggunakan metode Jackknife menghasilkan rata-rata sebesar 5,32, sedangkan metode Bootstrapping dengan 1.000 iterasi menghasilkan rata-rata sebesar 5,32139.

• Hasil dari kedua metode resampling tersebut menunjukkan nilai yang sangat dekat dengan rata-rata sampel awal. Dengan demikian, proses Jackknife dan Bootstrapping memberikan hasil estimasi rata-rata yang mendekati estimasi awal berdasarkan data yang digunakan.